# Förberedelser

## Importera

In [1]:
import SPARQLWrapper as sparql
import pandas as pd

In [2]:
endpoint = "https://libris.kb.se/sparql"

# Analys

## Digitiseringar
Den digitiserade versionen av en fysisk resurs. 

I dagsläget finns en hänvisning, som lokal entitet, till det fysiska originalet i otherPhysicalFormat > describedBy > controlNumber .

Målet är att byta ut den lokala entiteten mot en länakd entitet i reproductionOf.

In [30]:
query = '''

PREFIX marc: <https://id.kb.se/marc/>
PREFIX : <https://id.kb.se/vocab/>

SELECT  ?instance ?sameAsIdentifier ?localEntityControlNumber ?marcDisplayText WHERE {

  ?instance a :DigitalResource ;
    :otherPhysicalFormat ?localEntity;
    :sameAs ?sameAsIdentifier .

  ?localEntity :describedBy [ a :Record;
     :controlNumber ?localEntityControlNumber ] ;
     marc:displayText ?marcDisplayText .

  FILTER(REGEX(?localEntityControlNumber, "^[0-9]+$"))

  FILTER isBLANK(?localEntity) .

  FILTER(regex(?marcDisplayText, "(originalversion|orginalversion)", "i"))

}

'''

digitizations = sparql.get_sparql_dataframe(endpoint, query)
digitizations.info()
digitizations.head(1)

<class 'pandas.DataFrame'>
RangeIndex: 37279 entries, 0 to 37278
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   instance                  37279 non-null  str  
 1   sameAsIdentifier          37279 non-null  str  
 2   localEntityControlNumber  37279 non-null  str  
 3   marcDisplayText           37279 non-null  str  
dtypes: str(4)
memory usage: 1.1 MB


,instance,sameAsIdentifier,localEntityControlNumber,marcDisplayText
0,https://libris.kb.se/7rkw6cxk3srnwc3#it,http://libris.kb.se/resource/bib/19434947,2419426,Originalversion:


In [4]:
# Spara eventuella dubbletter för vidare analys. Spara i https://kbse.atlassian.net/browse/LXL-4790
# digitizations[digitizations.duplicated(keep=False)].to_csv("digitala_med_osynliga_blanknoder.csv")

In [31]:
# Preppa/städa
digitizations.drop_duplicates(inplace=True)
digitizations["sameAsIdentifier"] = digitizations["sameAsIdentifier"].str.lstrip("http://libris.kb.se/resource/bib/")

digitizations.info()
digitizations.head(1)

<class 'pandas.DataFrame'>
Index: 37272 entries, 0 to 37278
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   instance                  37272 non-null  str  
 1   sameAsIdentifier          37272 non-null  str  
 2   localEntityControlNumber  37272 non-null  str  
 3   marcDisplayText           37272 non-null  str  
dtypes: str(4)
memory usage: 1.4 MB


,instance,sameAsIdentifier,localEntityControlNumber,marcDisplayText
0,https://libris.kb.se/7rkw6cxk3srnwc3#it,19434947,2419426,Originalversion:


### Finns det digitiseringar med flera original? 🤯

In [35]:
repros_with_multiple_originals = digitizations[digitizations.duplicated(subset=["instance"], keep=False)]
repros_with_multiple_originals.value_counts("instance")

instance
https://libris.kb.se/5phq41ch47w3cxl#it    2
https://libris.kb.se/h1t2gj0t4jknmgg#it    2
https://libris.kb.se/h1t2wj4t55ctmvp#it    2
https://libris.kb.se/6qjz3wtj5j7lnc3#it    2
https://libris.kb.se/0jbqj6hb27c7kx3#it    2
Name: count, dtype: int64

Droppa dessa, för att istället hantera manuellt.

In [36]:
digitizations.drop_duplicates(subset="instance", keep=False, inplace=True)
digitizations.info()

<class 'pandas.DataFrame'>
Index: 37262 entries, 0 to 37278
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   instance                  37262 non-null  str  
 1   sameAsIdentifier          37262 non-null  str  
 2   localEntityControlNumber  37262 non-null  str  
 3   marcDisplayText           37262 non-null  str  
dtypes: str(4)
memory usage: 1.4 MB


In [8]:
#repros_with_multiple_originals.to_csv("reproduktioner_med_flera_original.tsv", sep="\t")

## Original
Den fysiska resurs som digitiserats.

In [37]:
query = '''

PREFIX marc: <https://id.kb.se/marc/>
PREFIX : <https://id.kb.se/vocab/>

SELECT ?instance ?sameAsIdentifier ?localEntityControlNumber ?marcDisplayText WHERE {

  ?instance a :PhysicalResource ;
    :otherPhysicalFormat ?localEntity;
    :sameAs ?sameAsIdentifier .

  FILTER isBLANK(?localEntity) .

  ?localEntity :describedBy [ a :Record;
     :controlNumber ?localEntityControlNumber ] ;
     marc:displayText ?marcDisplayText .

  FILTER(REGEX(?localEntityControlNumber, "^[0-9]+$"))
  FILTER(regex(?marcDisplayText, "(^digit.*version)", "i"))
  FILTER(!regex(?note, "(del|1969)", "i"))

}

'''

originals = sparql.get_sparql_dataframe(endpoint, query)
originals.info()
originals.head(1)

<class 'pandas.DataFrame'>
RangeIndex: 36497 entries, 0 to 36496
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   instance                  36497 non-null  str  
 1   sameAsIdentifier          36497 non-null  str  
 2   localEntityControlNumber  36497 non-null  str  
 3   marcDisplayText           36497 non-null  str  
dtypes: str(4)
memory usage: 1.1 MB


,instance,sameAsIdentifier,localEntityControlNumber,marcDisplayText
0,https://libris.kb.se/5phjr83h486wrh1#it,http://libris.kb.se/resource/bib/11847255,20883851,Digitaliserad version


In [ ]:
# Spara eventuella dubbletter för vidare analys. Spara i https://kbse.atlassian.net/browse/LXL-4790
#originals[originals.duplicated(keep=False)].to_csv("fysiska_med_osynliga_blanknoder.csv")

In [39]:
# Preppa/städa
originals.drop_duplicates(inplace=True)
originals["sameAsIdentifier"] = originals["sameAsIdentifier"].str.lstrip("http://libris.kb.se/resource/bib/")

originals.info()
originals.head(1)

<class 'pandas.DataFrame'>
Index: 36445 entries, 0 to 36496
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   instance                  36445 non-null  str  
 1   sameAsIdentifier          36445 non-null  str  
 2   localEntityControlNumber  36445 non-null  str  
 3   marcDisplayText           36445 non-null  str  
dtypes: str(4)
memory usage: 1.4 MB


,instance,sameAsIdentifier,localEntityControlNumber,marcDisplayText
0,https://libris.kb.se/5phjr83h486wrh1#it,11847255,20883851,Digitaliserad version


### Finns det original med flera digitiseringar?



In [45]:
originals_with_multiple_repros = originals[originals.duplicated(subset=["instance"], keep=False)]
originals_with_multiple_repros.info()
originals_with_multiple_repros.value_counts("instance")

<class 'pandas.DataFrame'>
Index: 441 entries, 144 to 36431
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   instance                  441 non-null    str  
 1   sameAsIdentifier          441 non-null    str  
 2   localEntityControlNumber  441 non-null    str  
 3   marcDisplayText           441 non-null    str  
dtypes: str(4)
memory usage: 17.2 KB


instance
https://libris.kb.se/xf742w281ml66nx#it    5
https://libris.kb.se/3ld3tvrf3dzzh56#it    4
https://libris.kb.se/zg8530895flm4dt#it    3
https://libris.kb.se/zg85cg3924sxpvc#it    3
https://libris.kb.se/r82pr8p332pl07z#it    3
                                          ..
https://libris.kb.se/xg8dh8g85fj89n0#it    2
https://libris.kb.se/l3wkp1bx5mbr473#it    2
https://libris.kb.se/h0shjjlt1j4lddp#it    2
https://libris.kb.se/h0sf34st1mm7xql#it    2
https://libris.kb.se/m4xltfhz1mr8sf9#it    2
Name: count, Length: 216, dtype: int64

## Jämförelse

In [ ]:
mutual_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["localEntityControlNumber", "sameAsIdentifier"],
    right_on=["sameAsIdentifier", "localEntityControlNumber"],
    suffixes=("_digital", "_physical")
)

# Helt identiska rader kan droppas
mutual_matches.drop_duplicates(inplace=True)

mutual_matches.info()
mutual_matches.head(5)

mutual_matches_to_print = mutual_matches.copy()
mutual_matches_to_print["instance_digital"] = mutual_matches_to_print["instance_digital"].str.extract(r"https://libris\.kb\.se/([a-zA-Z0-9]+)#it", expand=False)
mutual_matches_to_print["instance_physical"] = mutual_matches_to_print["instance_physical"].str.extract(r"https://libris\.kb\.se/([a-zA-Z0-9]+)#it", expand=False)
mutual_matches_to_print.to_csv("mutual_matches_ids.tsv", sep="\t", index=False)

<class 'pandas.DataFrame'>
RangeIndex: 35688 entries, 0 to 35687
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   instance_digital                   35688 non-null  str  
 1   sameAsIdentifier_digital           35688 non-null  str  
 2   localEntityControlNumber_digital   35688 non-null  str  
 3   marcDisplayText_digital            35688 non-null  str  
 4   instance_physical                  35688 non-null  str  
 5   sameAsIdentifier_physical          35688 non-null  str  
 6   localEntityControlNumber_physical  35688 non-null  str  
 7   marcDisplayText_physical           35688 non-null  str  
dtypes: str(8)
memory usage: 2.2 MB


In [46]:
# Vilka av de ömsesidigt matchande originalen har flera reproduktioner?
mutual_matches[mutual_matches.duplicated(subset=["instance_physical"], keep=False)]

,instance_digital,sameAsIdentifier_digital,localEntityControlNumber_digital,marcDisplayText_digital,instance_physical,sameAsIdentifier_physical,localEntityControlNumber_physical,marcDisplayText_physical
90,https://libris.kb.se/7rklfxvk02pc7p9#it,12322187,2437452,Originalversion:,https://libris.kb.se/2kc12nnd02sqfww#it,2437452,12322187,Digitaliserad version:
157,https://libris.kb.se/6qjw01hj21r0xh8#it,20102386,3324380,Originalversion:,https://libris.kb.se/btmbfrdn4qgjc83#it,3324380,20102386,Digitaliserad version:
515,https://libris.kb.se/g0s3xkcs31psqr7#it,18219234,2514674,Originalversion,https://libris.kb.se/4mf37ljg2x9tv6r#it,2514674,18219234,Digitaliserad version
654,https://libris.kb.se/xg8m9n185mrfm8x#it,19748498,2520856,Originalversion,https://libris.kb.se/6ph59vgj27bdcrv#it,2520856,19748498,Digitaliserad version
804,https://libris.kb.se/6qjnqh6j2bsnb45#it,14228716,2441182,Originalversion:,https://libris.kb.se/dwpcd44q0q7f4tz#it,2441182,14228716,Digitaliserad version:
...,...,...,...,...,...,...,...,...
35471,https://libris.kb.se/3mfwxl0f4d2rcp8#it,22550023,1619108,Originalversion:,https://libris.kb.se/xf7vw7882rph28d#it,1619108,22550023,Digitaliserad version:
35485,https://libris.kb.se/7rk12qpk58ltkz0#it,22549637,8854471,Originalversion:,https://libris.kb.se/p60xlzb155mbrd6#it,8854471,22549637,Digitaliserad version:
35558,https://libris.kb.se/xg8qwjs80jfpmss#it,22660688,1857890,Originalversion:,https://libris.kb.se/btm8lgzn333b1h5#it,1857890,22660688,Digitaliserad version:
35587,https://libris.kb.se/k3wcj5fw1s4v4b6#it,22660707,8854471,Originalversion:,https://libris.kb.se/p60xlzb155mbrd6#it,8854471,22660707,Digitaliserad version:


In [56]:
# Vilka reproduktioner har flera original?
mutual_matches[mutual_matches.duplicated(subset=["instance_digital"], keep=False)]

,instance_digital,sameAsIdentifier_digital,localEntityControlNumber_digital,marcDisplayText_digital,instance_physical,sameAsIdentifier_physical,localEntityControlNumber_physical,marcDisplayText_physical


## Ensidiga matcher

In [48]:
digi_to_physical_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["localEntityControlNumber"],
    right_on=["sameAsIdentifier"],
    suffixes=("_digital", "_physical")
)

digi_to_physical_matches.drop_duplicates(inplace=True)

digi_to_physical_matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 36265 entries, 0 to 36264
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   instance_digital                   36265 non-null  str  
 1   sameAsIdentifier_digital           36265 non-null  str  
 2   localEntityControlNumber_digital   36265 non-null  str  
 3   marcDisplayText_digital            36265 non-null  str  
 4   instance_physical                  36265 non-null  str  
 5   sameAsIdentifier_physical          36265 non-null  str  
 6   localEntityControlNumber_physical  36265 non-null  str  
 7   marcDisplayText_physical           36265 non-null  str  
dtypes: str(8)
memory usage: 2.2 MB


In [49]:
physical_to_digi_matches = digitizations.merge(
    originals,
    how="inner",
    left_on=["sameAsIdentifier"],
    right_on=["localEntityControlNumber"],
    suffixes=("_digital", "_physical")
)

physical_to_digi_matches.drop_duplicates(inplace=True)

physical_to_digi_matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 35804 entries, 0 to 35803
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   instance_digital                   35804 non-null  str  
 1   sameAsIdentifier_digital           35804 non-null  str  
 2   localEntityControlNumber_digital   35804 non-null  str  
 3   marcDisplayText_digital            35804 non-null  str  
 4   instance_physical                  35804 non-null  str  
 5   sameAsIdentifier_physical          35804 non-null  str  
 6   localEntityControlNumber_physical  35804 non-null  str  
 7   marcDisplayText_physical           35804 non-null  str  
dtypes: str(8)
memory usage: 2.2 MB


In [50]:
non_mutual_matches = pd.concat([physical_to_digi_matches, digi_to_physical_matches]).drop_duplicates(keep=False)
non_mutual_matches.info()

<class 'pandas.DataFrame'>
Index: 693 entries, 8 to 36250
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   instance_digital                   693 non-null    str  
 1   sameAsIdentifier_digital           693 non-null    str  
 2   localEntityControlNumber_digital   693 non-null    str  
 3   marcDisplayText_digital            693 non-null    str  
 4   instance_physical                  693 non-null    str  
 5   sameAsIdentifier_physical          693 non-null    str  
 6   localEntityControlNumber_physical  693 non-null    str  
 7   marcDisplayText_physical           693 non-null    str  
dtypes: str(8)
memory usage: 48.7 KB


In [51]:
non_mutual_matches.to_csv("non_mutual_matches.tsv", sep="\t")

In [52]:
non_mutual_matches["instance_digital"].value_counts()

instance_digital
https://libris.kb.se/j2vbjjtv3r846hn#it    23
https://libris.kb.se/dxq1sz9q2j2hxb2#it     4
https://libris.kb.se/cwp0rx8p5nnq792#it     4
https://libris.kb.se/g0s3v1cs33dpql7#it     4
https://libris.kb.se/h1t4w2dt5hx3lv9#it     4
                                           ..
https://libris.kb.se/m5zcfk1z1cmgt9w#it     1
https://libris.kb.se/xg8qwjs80jfpmss#it     1
https://libris.kb.se/k3wcj5fw1s4v4b6#it     1
https://libris.kb.se/k3wcd20w4z3896s#it     1
https://libris.kb.se/2ldpnfcd3mbq5f8#it     1
Name: count, Length: 624, dtype: int64

In [53]:
non_mutual_matches

,instance_digital,sameAsIdentifier_digital,localEntityControlNumber_digital,marcDisplayText_digital,instance_physical,sameAsIdentifier_physical,localEntityControlNumber_physical,marcDisplayText_physical
8,https://libris.kb.se/sb4572m41n6p35m#it,11716654,11716156,Originalversion:,https://libris.kb.se/p7124zj11973jnl#it,11716651,11716654,Digitaliserad version:
115,https://libris.kb.se/vc5520s64626ldv#it,10714146,10264683,Originalversion:,https://libris.kb.se/4mffrh0g4hs6gzs#it,10234394,10714146,Digitaliserad version:
116,https://libris.kb.se/vc5520s64626ldv#it,10714146,10264683,Originalversion:,https://libris.kb.se/8rkkd0nl1cwblxd#it,10648878,10714146,Digitaliserad version:
790,https://libris.kb.se/q8268w920zsg2sx#it,14224712,14224712,Originalversion:,https://libris.kb.se/xf74b9v8153sd2p#it,7615448,14224712,Digitaliserad version:
1065,https://libris.kb.se/vd687ls60lsnrjh#it,12458346,7619263,Originalversion,https://libris.kb.se/2kc8gk7d40c5q5k#it,7619262,12458346,Digitaliserad version
...,...,...,...,...,...,...,...,...
36094,https://libris.kb.se/m5zcfk1z1cmgt9w#it,20914289,2322981,Originalversion:,https://libris.kb.se/cvnb7qrp18j88t9#it,2322981,20914312,Digitaliserad version:
36131,https://libris.kb.se/xg8qwjs80jfpmss#it,22660688,1857890,Originalversion:,https://libris.kb.se/btm8lgzn333b1h5#it,1857890,13610234,Digitaliserad version:
36162,https://libris.kb.se/k3wcj5fw1s4v4b6#it,22660707,8854471,Originalversion:,https://libris.kb.se/p60xlzb155mbrd6#it,8854471,22549637,Digitaliserad version:
36175,https://libris.kb.se/k3wcd20w4z3896s#it,22549617,1644051,Originalversion:,https://libris.kb.se/cvn9ckdp1lr0cd3#it,1644051,22660728,Digitaliserad version:


In [55]:
non_mutual_matches[non_mutual_matches["instance_digital"] == "https://libris.kb.se/h1t2wj4t55ctmvp#it"]

,instance_digital,sameAsIdentifier_digital,localEntityControlNumber_digital,marcDisplayText_digital,instance_physical,sameAsIdentifier_physical,localEntityControlNumber_physical,marcDisplayText_physical
25936,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion,https://libris.kb.se/3ld219pf4l2dqzz#it,2399653,22698896,Digitaliserad version
25937,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion,https://libris.kb.se/8rk76g0l28x10gb#it,2399778,22698896,Digitaliserad version
25938,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion,https://libris.kb.se/3ld2192f2rk86qh#it,2399983,22698896,Digitaliserad version
25939,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion,https://libris.kb.se/1jb0z7mc0xxnnpk#it,2399651,22698896,Digitaliserad version
25940,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion,https://libris.kb.se/5ng43cwh0n95qkz#it,2399775,22698896,Digitaliserad version
25941,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion,https://libris.kb.se/2kc1081d09nwtv5#it,2399982,22698896,Digitaliserad version
25942,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion,https://libris.kb.se/btm98j0n0cbshs5#it,2399720,22698896,Digitaliserad version
25943,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion,https://libris.kb.se/2kc108nd425mclf#it,2399652,22698896,Digitaliserad version
25944,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion,https://libris.kb.se/2kc108sd30ggffg#it,2399772,22698896,Digitaliserad version
25946,https://libris.kb.se/j2vbjjtv3r846hn#it,22698896,3121391,Originalversion,https://libris.kb.se/9sl87hzm1hwt3l6#it,2399719,22698896,Digitaliserad version
